In [ ]:
"""
Part 2, Code 1: Advanced Cold Chain Spoilage Risk Model
=======================================================
Realistic model incorporating:
1. Time-varying temperature with diurnal cycles
2. Multiple temperature sensors with measurement error
3. Cumulative thermal damage (not just threshold exceedance)
4. Location variability within the truck
5. Door openings and recovery periods
6. Different vaccine types with varying sensitivity
7. Time-dependent spoilage kinetics (Arrhenius model)
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, beta
from scipy.optimize import minimize
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ============================================================
# 1. ADVANCED TEMPERATURE MODEL
# ============================================================

class TemperatureModel:
    """Realistic temperature model with multiple components"""
    
    def __init__(self, base_temp=4.0, ambient_temp=25.0):
        self.base_temp = base_temp
        self.ambient_temp = ambient_temp
        
    def generate_profile(self, duration_hours, dt_minutes=5, 
                         door_openings=None, truck_stops=None):
        """
        Generate realistic temperature profile
        """
        n_steps = int(duration_hours * 60 / dt_minutes)
        time = np.linspace(0, duration_hours, n_steps)
        
        # 1. Base refrigeration cycle (sinusoidal)
        refrigeration = 0.5 * np.sin(2 * np.pi * time / 0.5)  # 30-min cycle
        
        # 2. Diurnal temperature variation
        diurnal = 2.0 * np.sin(2 * np.pi * time / 24 - np.pi/4)
        
        # 3. Ambient influence (truck insulation imperfect)
        ambient_influence = 0.3 * (self.ambient_temp - self.base_temp) * \
                           (1 - np.exp(-time / 2))
        
        # 4. Door openings (if specified)
        door_effects = np.zeros_like(time)
        if door_openings is not None:
            for opening_time, duration, outdoor_temp in door_openings:
                idx = int(opening_time * 60 / dt_minutes)
                duration_steps = int(duration * 60 / dt_minutes)
                end_idx = min(idx + duration_steps, n_steps)
                # Temperature spike during door opening
                spike = (outdoor_temp - self.base_temp) * \
                        0.5 * np.exp(-np.linspace(0, 3, end_idx - idx))
                door_effects[idx:end_idx] = np.maximum(
                    door_effects[idx:end_idx], spike
                )
        
        # 5. Truck stops (engine off)
        stop_effects = np.zeros_like(time)
        if truck_stops is not None:
            for stop_time, duration in truck_stops:
                idx = int(stop_time * 60 / dt_minutes)
                duration_steps = int(duration * 60 / dt_minutes)
                end_idx = min(idx + duration_steps, n_steps)
                # Gradual warming during stops
                warm_up = (self.ambient_temp - self.base_temp) * \
                          0.1 * (1 - np.exp(-np.linspace(0, 2, end_idx - idx)))
                stop_effects[idx:end_idx] = warm_up
        
        # 6. Sensor noise
        sensor_noise = np.random.normal(0, 0.3, n_steps)
        
        # 7. Location variability (hotspots in truck)
        location_offset = np.random.normal(0, 0.5)  # Different positions in truck
        
        # Combine all effects
        temperature = (self.base_temp + location_offset + 
                      refrigeration + diurnal + ambient_influence + 
                      door_effects + stop_effects + sensor_noise)
        
        # Ensure physical bounds (can't go below 0°C or above 30°C)
        temperature = np.clip(temperature, -10, 30)
        
        return time, temperature

# ============================================================
# 2. VACCINE SPOILAGE KINETICS (Arrhenius Model)
# ============================================================

class VaccineKinetics:
    """
    Temperature-dependent degradation using Arrhenius equation
    Different vaccines have different sensitivity
    """
    
    def __init__(self, vaccine_type='mRNA'):
        # Parameters from literature (simplified)
        self.vaccine_params = {
            'mRNA': {'Ea': 90000, 'A': 1e12, 'T_ref': 4, 'critical_damage': 0.3},
            'protein': {'Ea': 70000, 'A': 1e9, 'T_ref': 4, 'critical_damage': 0.4},
            'viral': {'Ea': 60000, 'A': 1e8, 'T_ref': 4, 'critical_damage': 0.5},
            'live_attenuated': {'Ea': 50000, 'A': 1e7, 'T_ref': 4, 'critical_damage': 0.6}
        }
        self.params = self.vaccine_params[vaccine_type]
        self.R = 8.314  # Gas constant
        
    def degradation_rate(self, temp_C):
        """Calculate degradation rate at given temperature"""
        temp_K = temp_C + 273.15
        T_ref_K = self.params['T_ref'] + 273.15
        
        # Arrhenius equation
        rate = self.params['A'] * np.exp(
            -self.params['Ea'] / (self.R * temp_K)
        )
        # Normalize to reference temperature
        rate_ref = self.params['A'] * np.exp(
            -self.params['Ea'] / (self.R * T_ref_K)
        )
        return rate / rate_ref
    
    def calculate_damage(self, temperature_profile, time):
        """Calculate cumulative thermal damage"""
        rates = np.array([self.degradation_rate(t) for t in temperature_profile])
        
        # Time-weighted damage accumulation
        dt = np.mean(np.diff(time))
        damage = np.cumsum(rates) * dt
        
        # Normalize damage
        max_damage = damage[-1]
        damage_normalized = damage / (max_damage + 1e-10)
        
        return damage_normalized

# ============================================================
# 3. MONTE CARLO SIMULATION WITH MULTIPLE SCENARIOS
# ============================================================

class ColdChainSimulator:
    """Comprehensive cold chain simulation"""
    
    def __init__(self, num_scenarios=10000, duration_hours=24):
        self.num_scenarios = num_scenarios
        self.duration_hours = duration_hours
        
        # Model parameters with uncertainty
        self.temp_model = TemperatureModel(-2.0, 25.0)
        self.vaccine_types = ['mRNA', 'protein', 'viral', 'live_attenuated']
        
    def generate_scenario(self):
        """Generate a single scenario with random parameters"""
        
        # Randomize ambient conditions
        ambient_temp = np.random.uniform(15, 35)  # Variable weather
        base_temp = np.random.normal(4, 0.5)  # Slight variation in setpoint
        
        # Randomize door openings
        num_openings = np.random.poisson(2)
        door_openings = []
        for _ in range(num_openings):
            open_time = np.random.uniform(0.5, self.duration_hours - 1)
            duration = np.random.exponential(5) / 60  # Minutes
            outdoor_temp = np.random.uniform(ambient_temp - 5, ambient_temp + 5)
            door_openings.append((open_time, duration, outdoor_temp))
        
        # Randomize truck stops
        num_stops = np.random.poisson(1)
        truck_stops = []
        for _ in range(num_stops):
            stop_time = np.random.uniform(1, self.duration_hours - 1)
            duration = np.random.uniform(10, 30) / 60  # Minutes
            truck_stops.append((stop_time, duration))
        
        # Generate temperature profile
        time, temp = self.temp_model.generate_profile(
            self.duration_hours, dt_minutes=5,
            door_openings=door_openings, truck_stops=truck_stops
        )
        
        # Calculate damage for each vaccine type
        damages = {}
        for vtype in self.vaccine_types:
            kinetics = VaccineKinetics(vtype)
            damage = kinetics.calculate_damage(temp, time)
            critical_damage = kinetics.params['critical_damage']
            damages[vtype] = {
                'damage': damage,
                'critical': critical_damage,
                'spoiled': damage[-1] > critical_damage
            }
        
        return {
            'time': time,
            'temperature': temp,
            'door_openings': door_openings,
            'truck_stops': truck_stops,
            'ambient_temp': ambient_temp,
            'damages': damages,
            'metadata': {
                'base_temp': base_temp,
                'num_openings': num_openings,
                'num_stops': num_stops
            }
        }
    
    def run_simulation(self):
        """Run Monte Carlo simulation"""
        results = []
        for i in range(self.num_scenarios):
            scenario = self.generate_scenario()
            results.append(scenario)
            
            # Progress indicator
            if (i + 1) % 1000 == 0:
                print(f"Simulated {i+1:,} scenarios...")
        
        return results

# ============================================================
# 4. ANALYSIS AND VISUALIZATION
# ============================================================

def analyze_results(results, vaccine_type='mRNA'):
    """Comprehensive analysis of simulation results"""
    
    # Extract spoilage probabilities
    spoilage_probs = {}
    for vtype in results[0]['damages'].keys():
        spoilage = [r['damages'][vtype]['spoiled'] for r in results]
        spoilage_probs[vtype] = np.mean(spoilage)
    
    # Temperature statistics
    all_temps = np.concatenate([r['temperature'] for r in results])
    temp_stats = {
        'mean': np.mean(all_temps),
        'std': np.std(all_temps),
        'max': np.max(all_temps),
        'min': np.min(all_temps),
        'above_8': np.mean(all_temps > 8),
        'above_15': np.mean(all_temps > 15)
    }
    
    # Damage statistics for specific vaccine
    damages = [r['damages'][vaccine_type]['damage'][-1] for r in results]
    damage_stats = {
        'mean': np.mean(damages),
        'std': np.std(damages),
        'max': np.max(damages),
        'percentile_90': np.percentile(damages, 90),
        'percentile_95': np.percentile(damages, 95)
    }
    
    return {
        'spoilage_probs': spoilage_probs,
        'temp_stats': temp_stats,
        'damage_stats': damage_stats,
        'damages': damages
    }

def create_visualizations(results, analysis, vaccine_type='mRNA'):
    """Create comprehensive visualizations"""
    
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Example temperature profile with damage
    ax1 = plt.subplot(3, 3, 1)
    example = results[0]
    ax1.plot(example['time'], example['temperature'], 'b-', linewidth=1.5)
    ax1.axhline(8, color='r', linestyle='--', alpha=0.7, label='Critical (8°C)')
    ax1.axhline(4, color='g', linestyle='--', alpha=0.7, label='Setpoint (4°C)')
    ax1.fill_between(example['time'], example['temperature'], 8,
                     where=(example['temperature'] > 8), alpha=0.3, color='red')
    ax1.set_xlabel('Time (hours)')
    ax1.set_ylabel('Temperature (°C)')
    ax1.set_title('Example Temperature Profile')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Damage accumulation
    ax2 = plt.subplot(3, 3, 2)
    for vtype in results[0]['damages'].keys():
        damage = results[0]['damages'][vtype]['damage']
        ax2.plot(example['time'], damage, label=vtype, linewidth=1.5)
        ax2.axhline(results[0]['damages'][vtype]['critical'], 
                   linestyle='--', alpha=0.5)
    ax2.set_xlabel('Time (hours)')
    ax2.set_ylabel('Cumulative Damage')
    ax2.set_title('Damage Accumulation by Vaccine Type')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Spoilage probability by vaccine type
    ax3 = plt.subplot(3, 3, 3)
    vtypes = list(analysis['spoilage_probs'].keys())
    probs = list(analysis['spoilage_probs'].values())
    colors = ['red', 'orange', 'yellow', 'green']
    bars = ax3.bar(vtypes, probs, color=colors, alpha=0.7)
    ax3.set_ylabel('Spoilage Probability')
    ax3.set_title('Spoilage Probability by Vaccine Type')
    ax3.set_ylim(0, max(probs) * 1.2)
    for bar, prob in zip(bars, probs):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{prob:.3f}', ha='center', va='bottom')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Temperature distribution
    ax4 = plt.subplot(3, 3, 4)
    all_temps = np.concatenate([r['temperature'] for r in results])
    ax4.hist(all_temps, bins=50, density=True, alpha=0.7, color='blue')
    ax4.axvline(8, color='r', linestyle='--', label='Critical')
    ax4.axvline(4, color='g', linestyle='--', label='Setpoint')
    ax4.set_xlabel('Temperature (°C)')
    ax4.set_ylabel('Density')
    ax4.set_title('Temperature Distribution (All Scenarios)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Damage distribution
    ax5 = plt.subplot(3, 3, 5)
    damages = analysis['damages']
    ax5.hist(damages, bins=50, density=True, alpha=0.7, color='purple')
    ax5.axvline(analysis['spoilage_probs'][vaccine_type], color='r', 
               linestyle='--', label='Spoilage threshold')
    ax5.set_xlabel('Final Damage')
    ax5.set_ylabel('Density')
    ax5.set_title(f'Damage Distribution ({vaccine_type} vaccine)')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # 6. Time above temperature thresholds
    ax6 = plt.subplot(3, 3, 6)
    threshold_times = []
    thresholds = [6, 8, 10, 12, 15]
    for thresh in thresholds:
        times_above = []
        for r in results[:1000]:  # Sample for speed
            time_above = np.sum(r['temperature'] > thresh) * 0.0833  # Convert to hours
            times_above.append(time_above)
        threshold_times.append(np.mean(times_above))
    
    ax6.bar([f'>{t}°C' for t in thresholds], threshold_times, alpha=0.7)
    ax6.set_xlabel('Temperature Threshold')
    ax6.set_ylabel('Time Above Threshold (hours)')
    ax6.set_title('Average Time Above Temperature Thresholds')
    ax6.grid(True, alpha=0.3, axis='y')
    
    # 7. Risk heatmap (ambient temp vs door openings)
    ax7 = plt.subplot(3, 3, 7)
    ambient_temps = [r['ambient_temp'] for r in results[:1000]]
    door_counts = [r['metadata']['num_openings'] for r in results[:1000]]
    spoilage = [r['damages'][vaccine_type]['spoiled'] for r in results[:1000]]
    
    # Create 2D histogram
    h, xedges, yedges = np.histogram2d(ambient_temps, door_counts, 
                                       bins=[20, 5], weights=spoilage)
    counts, _, _ = np.histogram2d(ambient_temps, door_counts, 
                                  bins=[20, 5])
    with np.errstate(divide='ignore', invalid='ignore'):
        risk = np.divide(h, counts, where=counts>0)
        risk[counts==0] = np.nan
    
    im = ax7.imshow(risk.T, origin='lower', aspect='auto',
                    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
                    cmap='RdYlGn_r', vmin=0, vmax=1)
    ax7.set_xlabel('Ambient Temperature (°C)')
    ax7.set_ylabel('Number of Door Openings')
    ax7.set_title(f'Risk Heatmap ({vaccine_type} vaccine)')
    plt.colorbar(im, ax=ax7, label='Spoilage Probability')
    
    # 8. Temperature profile example with annotations
    ax8 = plt.subplot(3, 3, 8)
    example = results[1]  # Another example
    ax8.plot(example['time'], example['temperature'], 'b-', linewidth=1.5)
    
    # Annotate door openings
    for opening in example['door_openings']:
        ax8.axvspan(opening[0], opening[0] + opening[1], 
                   alpha=0.2, color='orange')
    # Annotate truck stops
    for stop in example['truck_stops']:
        ax8.axvspan(stop[0], stop[0] + stop[1], 
                   alpha=0.2, color='gray')
    
    ax8.set_xlabel('Time (hours)')
    ax8.set_ylabel('Temperature (°C)')
    ax8.set_title('Temperature Profile with Events')
    ax8.legend(['Temperature', 'Door Opening', 'Truck Stop'])
    ax8.grid(True, alpha=0.3)
    
    # 9. Statistical summary table
    ax9 = plt.subplot(3, 3, 9)
    ax9.axis('off')
    
    # Create summary text
    summary_text = f"""
    SIMULATION SUMMARY
    ==================
    Total Scenarios: {len(results):,}
    Vaccine Analyzed: {vaccine_type}
    
    Temperature Statistics:
    • Mean: {analysis['temp_stats']['mean']:.2f}°C
    • Std Dev: {analysis['temp_stats']['std']:.2f}°C
    • Max: {analysis['temp_stats']['max']:.2f}°C
    • P(T > 8°C): {analysis['temp_stats']['above_8']:.3f}
    • P(T > 15°C): {analysis['temp_stats']['above_15']:.3f}
    
    Damage Statistics ({vaccine_type}):
    • Mean Damage: {analysis['damage_stats']['mean']:.3f}
    • 95th Percentile: {analysis['damage_stats']['percentile_95']:.3f}
    • Spoilage Risk: {analysis['spoilage_probs'][vaccine_type]:.3f}
    
    Risk by Vaccine Type:
    """
    for vtype, prob in analysis['spoilage_probs'].items():
        summary_text += f"\n    • {vtype}: {prob:.3f}"
    
    ax9.text(0.1, 0.9, summary_text, transform=ax9.transAxes,
            fontsize=9, verticalalignment='top', fontfamily='monospace')
    
    plt.tight_layout()
    plt.show()

# ============================================================
# 5. MAIN EXECUTION
# ============================================================

def main():
    print("=" * 70)
    print("ADVANCED COLD CHAIN SPOILAGE RISK MODEL")
    print("=" * 70)
    print("This simulation incorporates:")
    print("  • Time-varying temperature with diurnal cycles")
    print("  • Multiple vaccine types with different sensitivity")
    print("  • Cumulative thermal damage (Arrhenius kinetics)")
    print("  • Door openings and truck stops")
    print("  • Location variability and sensor noise")
    print("  • Uncertainty in ambient conditions")
    print("=" * 70)
    
    # Run simulation
    print("\nStarting simulation...")
    simulator = ColdChainSimulator(num_scenarios=5000, duration_hours=12)
    results = simulator.run_simulation()
    
    # Analyze results
    print("\nAnalyzing results...")
    analysis = analyze_results(results, vaccine_type='mRNA')
    
    # Print key results
    print("\n" + "=" * 70)
    print("KEY RESULTS")
    print("=" * 70)
    print(f"Temperature Statistics:")
    print(f"  Mean: {analysis['temp_stats']['mean']:.2f}°C")
    print(f"  Std Dev: {analysis['temp_stats']['std']:.2f}°C")
    print(f"  P(T > 8°C): {analysis['temp_stats']['above_8']:.3f}")
    print(f"\nSpoilage Probabilities:")
    for vtype, prob in analysis['spoilage_probs'].items():
        print(f"  {vtype}: {prob:.4f}")
    print(f"\nmRNA Vaccine Damage Statistics:")
    print(f"  Mean Damage: {analysis['damage_stats']['mean']:.3f}")
    print(f"  95th Percentile: {analysis['damage_stats']['percentile_95']:.3f}")
    print(f"  Max Damage: {analysis['damage_stats']['max']:.3f}")
    
    # Create visualizations
    print("\nGenerating visualizations...")
    create_visualizations(results, analysis, vaccine_type='mRNA')
    
    print("\nSimulation complete!")
    print("=" * 70)

if __name__ == "__main__":
    main()